In [65]:
import pandas as pd
import numpy as np

### Load Data

In [66]:
# Load the data
btc = pd.read_csv("BTC_full_data.csv")
eth = pd.read_csv("ETH_full_data.csv")

print(btc.head())
print(eth.head())

         Date          Open          High           Low         Close  \
0  2018-01-01  14112.200195  14112.200195  13154.700195  13657.200195   
1  2018-01-02  13625.000000  15444.599609  13163.599609  14982.099609   
2  2018-01-03  14978.200195  15572.799805  14844.500000  15201.000000   
3  2018-01-04  15270.700195  15739.700195  14522.200195  15599.200195   
4  2018-01-05  15477.200195  17705.199219  15202.799805  17429.500000   

      Adj Close       Volume  Log_Return  
0  13657.200195  10291200000         NaN  
1  14982.099609  16846600192    0.092589  
2  15201.000000  16871900160    0.014505  
3  15599.200195  21783199744    0.025858  
4  17429.500000  23840899072    0.110945  
         Date        Open         High         Low       Close   Adj Close  \
0  2018-01-01  755.757019   782.530029  742.004028  772.640991  772.640991   
1  2018-01-02  772.346008   914.830017  772.346008  884.443970  884.443970   
2  2018-01-03  886.000000   974.471008  868.450989  962.719971  962.7

In [67]:
def bollinger_mean_reversion(df, window=20, num_std=2):
    df = df.copy()

    df["ma"] = df["Close"].rolling(window).mean()
    df["std"] = df["Close"].rolling(window).std()
    df["upper_band"] = df["ma"] + num_std * df["std"]
    df["lower_band"] = df["ma"] - num_std * df["std"]

    position = []
    current_pos = 0

    for i in range(len(df)):
        close = df["Close"].iloc[i]
        ma = df["ma"].iloc[i]
        upper = df["upper_band"].iloc[i]
        lower = df["lower_band"].iloc[i]

        if np.isnan(ma) or np.isnan(upper) or np.isnan(lower):
            position.append(0)
            continue

        if current_pos == 0:
            if close < lower:
                current_pos = 1
            elif close > upper:
                current_pos = -1
        elif current_pos == 1:
            if close >= ma:
                current_pos = 0
        elif current_pos == -1:
            if close <= ma:
                current_pos = 0

        position.append(current_pos)

    df["position"] = position
    df["strategy_return"] = df["position"].shift(1) * df["Log_Return"]

    return df

In [68]:
btc_bb = bollinger_mean_reversion(btc, window=20, num_std=2)
eth_bb = bollinger_mean_reversion(eth, window=20, num_std=2)

### Evaluation
We will evaluate the strategy using 6 metrics:
1. Cumulative PnL
2. Average Daily PnL
3. Volatility
4. Annualised Return
5. Sharpe Ratio
6. Max Drawdown

In [69]:
def evaluate_strategy(strategy_returns):
    r = strategy_returns.dropna()

    # cumulative pnl
    cumulative_pnl = np.exp(r.cumsum()).iloc[-1] - 1
    # average daily pnl
    avg_daily_pnl = r.mean()
    # volatility
    volatility = r.std()
    # annualised return
    annual_return = r.mean() * 365
    # annualised volatility
    annual_vol = volatility * np.sqrt(365)
    # sharpe ratio
    rf = 0.03
    sharpe = (annual_return - rf) / annual_vol
    # max drawdown
    cumulative = np.exp(r.cumsum())
    peak = cumulative.cummax()
    drawdown = (cumulative - peak) / peak
    max_dd = drawdown.min()

    return {
        "Cumulative PnL": cumulative_pnl,
        "Average Daily PnL": avg_daily_pnl,
        "Volatility": volatility,
        "Annualised Return": annual_return,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd
    }


In [70]:
# Run evaluation
btc_results = evaluate_strategy(btc_bb["strategy_return"])
eth_results = evaluate_strategy(eth_bb["strategy_return"])

print("BTC Results")
print(btc_results)

print("\nETH Results")
print(eth_results)

BTC Results
{'Cumulative PnL': np.float64(-0.7748770640708358), 'Average Daily PnL': np.float64(-0.0005106536452337564), 'Volatility': np.float64(0.026885707127006206), 'Annualised Return': np.float64(-0.1863885805103211), 'Sharpe Ratio': np.float64(-0.4212757324157189), 'Max Drawdown': np.float64(-0.9250128838783174)}

ETH Results
{'Cumulative PnL': np.float64(-0.9494640893613103), 'Average Daily PnL': np.float64(-0.0010222846211156349), 'Volatility': np.float64(0.034581810327124846), 'Annualised Return': np.float64(-0.3731338867072067), 'Sharpe Ratio': np.float64(-0.6101760385395893), 'Max Drawdown': np.float64(-0.9701293923184195)}


### Results Interpretation
BTC:
1. Cumulative PnL: -0.77 -> The portfolio lost 77 times its initial value
2. Average Daily PnL: -0.051% avarage return per day
3. Volatility: 2.69%
4. Annualised Return: -0.186 -> 19% per year
5. Sharpe Ratio: -0.42
6. Max Drawdown: -0.92

ETH:
1. Cumulative PnL: -0.94 -> The portfolio lost 94 times its initial value
2. Average Daily PnL: -0.001% avarage return per day
3. Volatility: 3.46%
4. Annualised Return: 0.37 -> 37% per year
5. Sharpe Ratio: -0.61
6. Max Drawdown: -0.97